In [1]:
import os
import mlflow
import joblib
import numpy as np
import pandas as pd
import mlflow.sklearn
from pathlib import Path
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

In [2]:
df = pd.read_csv(r'C:\Users\priya\Desktop\PyCh_Pro\Churn_Analysis_and_Modelling\data\raw\telco_churn.csv')

In [4]:
df = df.drop(columns = ["customerID"])

In [5]:
df.sample(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
2149,Female,0,No,No,46,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,One year,Yes,Electronic check,101.10,4674.4,No
4019,Male,0,No,No,14,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.55,1415.55,Yes
1990,Female,1,No,No,16,Yes,Yes,Fiber optic,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,84.75,1350.15,Yes
5130,Male,0,No,No,8,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,30.45,226.45,Yes
3489,Female,0,Yes,No,26,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,One year,Yes,Credit card (automatic),90.10,2312.55,No


## Started Preparing Data

In [6]:
from sklearn.preprocessing import FunctionTransformer

def preprocessing_raw_data(X):
    df = X.copy()

    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
        
    internet_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
    ]
    phone_cols = ["MultipleLines"]
    binary_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies",
        "MultipleLines", "Partner", "Dependents",
        "PhoneService", "PaperlessBilling"
    ]

    
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()
            
    for col in internet_cols:
        if col in df.columns:
            df[col] = df[col].replace("no internet service", "no")
            
    for col in phone_cols:
        if col in df.columns:
            df[col] = df[col].replace("no phone service", "no")
            
    df["Stability"] = df["Partner"].astype(str) + "_" + df["Dependents"].astype(str)
    
    return df

new_feature_clean_transformer = FunctionTransformer(preprocessing_raw_data)

In [7]:
def binaryEncoder(X):
    df = X.copy()
    binary_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies",
        "MultipleLines", "Partner", "Dependents",
        "PhoneService", "PaperlessBilling"
    ]
    
    mapping = {"no": 0, "yes": 1}
    
    for col in binary_cols:
        if col in df.columns:
            # map it, and used fillna just in case a weird value sneaks in
            df[col] = df[col].map(mapping).fillna(df[col])
            
    return df

binary_encoder_transformer = FunctionTransformer(binaryEncoder)

In [8]:
def precision_recall_at_k(y_test, y_prob, return_precision = True, k = 0.2):
    """
    Compute Recall@K and Precision@K for ranking-based evaluation.
    
    Parameters:
    - y_test: true labels (0/1)
    - y_prob: predicted probabilities
    - k: fraction of top samples to consider
    
    Returns:
    - recall_at_k, precision_at_k
    """
    df_eval = pd.DataFrame({
        "prob" : y_prob,
        "actual" : y_test.values
    })
    df_eval = df_eval.sort_values(by = "prob", ascending = False)

    top_k = max(1, int(k * len(df_eval)))
    top_churners = df_eval.head(top_k)
    
    total_churn = df_eval["actual"].sum()
    if total_churn == 0:
        return (0, 0) if return_precision else 0
    recall = top_churners["actual"].sum() / total_churn
    # out of all the churners, how many churns are catched in the top 20%
    if return_precision:
        precision = top_churners["actual"].mean() if len(top_churners) > 0 else 0
        # in the top 20% range, how many are actual churns.
        return recall, precision

    return recall

## Building pipeline for Classification

In [5]:
X = df.drop(columns = ["Churn"]) # shape(7043, 19)
y = df["Churn"] # shape(7043,)

y_mapped = df["Churn"].map({"Yes": 1, "No": 0})

# first split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_mapped, test_size=0.30, stratify=y, random_state=42
)

# Second split
X_test, X_stream, y_test, y_stream = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Data Split Complete:")
print(f"Training data: {X_train.shape[0]} rows (70%)")
print(f"Testing data:  {X_test.shape[0]} rows (15%)")
print(f"Stream data:   {X_stream.shape[0]} rows (15%)")



Data Split Complete:
Training data: 4930 rows (70%)
Testing data:  1056 rows (15%)
Stream data:   1057 rows (15%)


In [18]:
import os
save_dir = "data/processed"
os.makedirs(save_dir, exist_ok=True)

In [19]:
batch1 = X_test[:500:1]
batch2 = X_test[:500:-1]
super_batch = X_test[:1000:1]

batch1.to_csv(f"{save_dir}/batch_test_sample_1.csv", index=False)
batch2.to_csv(f"{save_dir}/batch_test_sample_2.csv", index=False)
super_batch.to_csv(f"{save_dir}/batch_test_sample_super.csv", index=False)

print(f"Success! Files saved in: {os.path.abspath(save_dir)}")

Success! Files saved in: C:\Users\priya\Desktop\PyCh_Pro\Churn_Analysis_and_Modelling\notebooks\data\processed


In [10]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

master_column_transformer = ColumnTransformer(
    transformers=[
        ("scaling", StandardScaler(), ["tenure", "MonthlyCharges", "TotalCharges"]),
        ("ohe", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), ["InternetService", "PaymentMethod", "gender", "Stability"]),
        ("ordinal", OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]]), ["Contract"])
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

In [12]:
model1 = RandomForestClassifier()
random_pipeline = Pipeline([
    ("new_feature_and_data_cleaning", new_feature_clean_transformer),
    ("binary_encoder", binary_encoder_transformer),
    ("Preprocessor", master_column_transformer),
    ("Model", model1)
])

In [13]:
model2 = XGBClassifier()
xgb_pipeline = Pipeline([
    ("new_feature_and_data_cleaning", new_feature_clean_transformer),
    ("binary_encoder", binary_encoder_transformer),
    ("Preprocessor", master_column_transformer),
    ("Model", model2)
])

In [18]:
print("--- 1. Connecting to the MLflow Vault ---")
master_path = Path("C:/Users/priya/Desktop/PyCh_Pro/Churn_Analysis_and_Modelling")
db_uri = f"sqlite:///{master_path.as_posix()}/mlflow.db"
artifact_uri = f"file:///{master_path.as_posix()}/mlruns" 

mlflow.set_tracking_uri(db_uri)
experiment_name = "Telecom_Churn_Final_Pipelines"

try:
    mlflow.create_experiment(name=experiment_name, artifact_location=artifact_uri)
except Exception:
    pass

mlflow.set_experiment(experiment_name)
print(f"Tracking URI locked to: {mlflow.get_tracking_uri()}")
print(f"Active Experiment: '{experiment_name}'\n")

ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])

print("--- 2. Injecting Winning Parameters ---")

# 1. Random Forest Pipeline
random_pipeline.set_params(
    Model__n_estimators=500,
    Model__max_depth=5,
    Model__min_samples_leaf=2,
    Model__min_samples_split=5,
    Model__class_weight="balanced",
    Model__random_state=42
)
print("SUCCESS: Random Forest parameters injected.")

# 2. XGBoost Pipeline
xgb_pipeline.set_params(
    Model__n_estimators=100,
    Model__learning_rate=0.05,
    Model__max_depth=3,
    Model__subsample=0.8,
    Model__scale_pos_weight=ratio, 
    Model__eval_metric="logloss",
    Model__random_state=42
)
print("SUCCESS: XGBoost parameters injected.\n")

print("--- 3. Starting the Final Pipeline Arena ---")
models_to_test = {
    "Production_RF_Pipeline": random_pipeline,
    "Production_XGB_Pipeline": xgb_pipeline
}

for model_name, model_pipeline in models_to_test.items():
    with mlflow.start_run(run_name=model_name):
        print(f"Training and Evaluating: {model_name}...")
        
        model_pipeline.fit(X_train, y_train)
        
        y_prob = model_pipeline.predict_proba(X_test)[:, 1]
        y_pred = model_pipeline.predict(X_test)
        
        roc_auc = roc_auc_score(y_test, y_prob)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        recall_20, precision_20 = precision_recall_at_k(y_test, y_prob) 
        
        mlflow.log_param("pipeline_type", model_name)
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("recall_20", recall_20)
        mlflow.log_metric("precision_20", precision_20)
        
        mlflow.sklearn.log_model(model_pipeline, "model")
        
        print(f"SUCCESS: Logged to MLflow | ROC-AUC: {roc_auc:.4f} | Recall: {recall:.4f} | Recall@20: {recall_20:.4f}\n")

print("🏆 Final Pipeline Test Complete!")

--- 1. Connecting to the MLflow Vault ---
Tracking URI locked to: sqlite:///C:/Users/priya/Desktop/PyCh_Pro/Churn_Analysis_and_Modelling/mlflow.db
Active Experiment: 'Telecom_Churn_Final_Pipelines'

--- 2. Injecting Winning Parameters ---
SUCCESS: Random Forest parameters injected.
SUCCESS: XGBoost parameters injected.

--- 3. Starting the Final Pipeline Arena ---
Training and Evaluating: Production_RF_Pipeline...


2026/04/14 15:57:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 15:57:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


SUCCESS: Logged to MLflow | ROC-AUC: 0.8473 | Recall: 0.8357 | Recall@20: 0.5143

Training and Evaluating: Production_XGB_Pipeline...


2026/04/14 15:58:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 15:58:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


SUCCESS: Logged to MLflow | ROC-AUC: 0.8487 | Recall: 0.8500 | Recall@20: 0.5036

🏆 Final Pipeline Test Complete!


### Model Saving Script

In [19]:
joblib.dump(xgb_pipeline, "production_pipeline.pkl") 

['production_pipeline.pkl']

### Saving Stream Data

In [16]:
stream_df = X_stream.copy()
stream_df['Churn'] = y_stream

# Export to the folder where your Streamlit app.py lives
stream_df.to_csv(r"C:\Users\priya\Desktop\PyCh_Pro\Churn_Analysis_and_Modelling\data\raw\stream_data.csv", index=False)